# E0 — Text-Only Sentiment Baseline

This notebook implements the **E0** control experiment for the research question:
> *Do emojis carry sentiment information that words miss?*

- **E0 uses `text_without_emoji`** (emojis explicitly removed) so the model cannot see emojis.
- Frozen `bert-base-uncased` → mean pooling → trainable MLP head → 3 classes (Bearish/Neutral/Bullish).
- Class-weighted cross-entropy (weights from train only).
- Run on a **T4 GPU** runtime in Google Colab.

## 1. Setup and Environment Configuration

Import the required libraries, load configuration, set the random seed, and verify GPU availability.

In [ ]:
# Cell: Setup and config
import os, sys, json, random, time, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

# Configuration
CFG = {
    "model_name": "bert-base-uncased",
    "max_length": 128,
    "seed": 42,
    "batch_size": 32,
    "epochs": 5,
    "learning_rate": 0.001,
    "weight_decay": 1e-4,
    "hidden_size": 768,
    "classifier_hidden": 256,
    "dropout": 0.3,
    "n_classes": 3,
    "class_names": ["Bearish", "Neutral", "Bullish"],
}

# Reproducibility
random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG["seed"])

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

# Output dir
os.makedirs("results/E0", exist_ok=True)
os.makedirs("data/processed/canonical", exist_ok=True)

## 2. Load and Validate Preprocessed Data

Load the final canonical CSV files and validate the expected row counts and columns.

In [ ]:
# Cell: Load and validate canonical data
# Load canonical JSONL files (use the final approved splits)
train_df = pd.read_json("data/processed/canonical/final_train.jsonl", lines=True)
val_df = pd.read_json("data/processed/canonical/final_validation.jsonl", lines=True)
test_df = pd.read_json("data/processed/canonical/final_test.jsonl", lines=True)

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

# Validate expected counts
expected = {"train": 91121, "validation": 20676, "test": 11966}
assert len(train_df) == expected["train"], "Train count mismatch"
assert len(val_df) == expected["validation"], "Validation count mismatch"
assert len(test_df) == expected["test"], "Test count mismatch"
print("Split counts validated OK")

# Validate text_without_emoji column
for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    assert "text_without_emoji" in df.columns, f"Missing column in {name}"
    assert df["text_without_emoji"].notna().all(), f"Null values in {name}"
print("text_without_emoji column present and non-null in all splits")

# Label validation
for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    assert set(df["label"].unique()) <= {0, 1, 2}, f"Invalid labels in {name}"
print("All labels valid (0/1/2)")

# Preview a row
print("\nSample row:")
print(train_df.iloc[0][["original_text", "text_without_emoji", "emoji_list", "label", "label_name"]].to_dict())

## 3. Data Exploration and Class Distribution Analysis

Compute class distribution, class weights (train only), and visualize the imbalance.

In [ ]:
# Cell: Class distribution and class weights
from collections import Counter

# Class distributions
train_dist = Counter(train_df["label"])
val_dist = Counter(val_df["label"])
test_dist = Counter(test_df["label"])

print("Train distribution:", {CFG["class_names"][k]: v for k, v in train_dist.items()})
print("Validation distribution:", {CFG["class_names"][k]: v for k, v in val_dist.items()})
print("Test distribution:", {CFG["class_names"][k]: v for k, v in test_dist.items()})

# Class weights from TRAIN ONLY (inverse frequency, normalized so mean=1)
total = sum(train_dist.values())
n_classes = CFG["n_classes"]
class_weights = {c: total / (n_classes * train_dist[c]) for c in range(3)}
print("\nClass weights (train-only):", {CFG["class_names"][k]: round(v, 4) for k, v in class_weights.items()})

# Plot class imbalance
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, dist) in zip(axes, [("Train", train_dist), ("Validation", val_dist), ("Test", test_dist)]):
    labels = [CFG["class_names"][i] for i in range(3)]
    counts = [dist.get(i, 0) for i in range(3)]
    sns.barplot(x=labels, y=counts, ax=ax, palette="viridis")
    ax.set_title(f"{name} Distribution")
    ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig("results/E0/class_distribution.png", dpi=150)
plt.show()
print("\nSaved class imbalance plot -> results/E0/class_distribution.png")

# Text length and emoji stats
print("\nToken count stats (train):")
print("  mean:", round(train_df["token_count"].mean(), 2))
print("  median:", train_df["token_count"].median())
print("  max:", train_df["token_count"].max())
print("Emoji count stats (train):")
print("  mean:", round(train_df["num_emojis"].mean(), 2))
print("  median:", train_df["num_emojis"].median())
print("  rows with 0 emojis:", (train_df["num_emojis"] == 0).sum())

## 4. BERT Featurization Pipeline

Load frozen `bert-base-uncased`, tokenize with `max_length=128`, and extract mean-pooled embeddings (cached).

In [ ]:
# Cell: Load frozen BERT
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
bert = AutoModel.from_pretrained(CFG["model_name"])
# Freeze encoder
for param in bert.parameters():
    param.requires_grad = False
bert.to(device)
bert.eval()
print("BERT loaded and frozen:", CFG["model_name"])

In [ ]:
# Cell: Featurize with mean pooling (cached)
def featurize(texts, tokenizer, bert, max_length, batch_size, device):
    """Extract mean-pooled BERT embeddings for a list of texts."""
    embeds = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=max_length,
                            return_tensors="pt").to(device)
            out = bert(**enc)
            last_hidden = out.last_hidden_state  # (B, L, H)
            mask = enc["attention_mask"].unsqueeze(-1).float()
            summed = (last_hidden * mask).sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1e-9)
            pooled = (summed / counts).cpu().numpy()
            embeds.append(pooled)
    return np.concatenate(embeds, axis=0)


CACHE_DIR = "results/E0/embeddings_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def load_or_featurize(df, split_name, tokenizer, bert, cfg):
    path = os.path.join(CACHE_DIR, f"embeds_{split_name}.npy")
    if os.path.exists(path):
        print(f"Loading cached embeddings: {split_name}")
        return np.load(path)
    print(f"Featurizing {split_name}: {len(df)} examples...")
    emb = featurize(df["text_without_emoji"].tolist(), tokenizer, bert,
                    cfg["max_length"], cfg["batch_size"], device)
    np.save(path, emb)
    print(f"Saved embeddings for {split_name} -> {path}")
    return emb

X_train = load_or_featurize(train_df, "train", tokenizer, bert, CFG)
X_val = load_or_featurize(val_df, "validation", tokenizer, bert, CFG)
X_test = load_or_featurize(test_df, "test", tokenizer, bert, CFG)

y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

print("Embedding shapes:", X_train.shape, X_val.shape, X_test.shape)

## 5. Model Architecture Definition

Define the MLP classification head (768 → 256 → 3) with dropout and ReLU.

In [ ]:
# Cell: Define MLP classification head
class MLPHead(nn.Module):
    """Trainable MLP classification head on frozen BERT embeddings."""
    def __init__(self, in_dim, hidden_dim, n_classes, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes),
        )
        # Xavier uniform init
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, x):
        return self.net(x)


model = MLPHead(CFG["hidden_size"], CFG["classifier_hidden"], CFG["n_classes"], CFG["dropout"])
model.to(device)
print("MLP head created:")
print(model)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 6. Training Loop with Class-Weighted Loss

Train the head with class-weighted cross-entropy, validate each epoch, and save the best model by validation macro-F1.

In [ ]:
# Cell: Training loop
# Class-weighted loss (weights from train only)
weights_tensor = torch.tensor([class_weights[c] for c in range(3)], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights_tensor)

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["learning_rate"], weight_decay=CFG["weight_decay"])

# Data loaders
train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                         torch.tensor(y_train, dtype=torch.long))
val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                       torch.tensor(y_val, dtype=torch.long))
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True)
val_loader = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False)

best_val_f1 = -1.0
best_epoch = -1
patience = 5
no_improve = 0
history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": []}
start = time.time()

for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    total_loss, n_batches = 0.0, 0
    for emb, lab in train_loader:
        emb, lab = emb.to(device), lab.to(device)
        optimizer.zero_grad()
        logits = model(emb)
        loss = criterion(logits, lab)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    train_loss = total_loss / max(n_batches, 1)

    # Validation
    model.eval()
    val_loss, preds, true = 0.0, [], []
    with torch.no_grad():
        for emb, lab in val_loader:
            emb, lab = emb.to(device), lab.to(device)
            logits = model(emb)
            val_loss += criterion(logits, lab).item()
            preds.extend(logits.argmax(dim=-1).cpu().numpy().tolist())
            true.extend(lab.cpu().numpy().tolist())
    val_loss /= max(len(val_loader), 1)
    val_acc = accuracy_score(true, preds)
    val_f1 = f1_score(true, preds, average="macro", zero_division=0)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    print(f"Epoch {epoch}/{CFG['epochs']} | TrainLoss {train_loss:.4f} | "
          f"ValLoss {val_loss:.4f} | ValAcc {val_acc:.4f} | ValMacroF1 {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        no_improve = 0
        torch.save({"state": model.state_dict(), "best_val_f1": val_f1,
                    "best_epoch": epoch, "config": CFG},
                   "results/E0/best_model.pt")
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

training_time = time.time() - start
print(f"\nTraining complete in {training_time:.1f}s. Best epoch: {best_epoch}, Best val macro-F1: {best_val_f1:.4f}")

## 7. Evaluation on Test Set

Load the best checkpoint, run inference on the test set, compute metrics, and build the confusion matrix.

In [ ]:
# Cell: Evaluate on test set
# Load best checkpoint
ckpt = torch.load("results/E0/best_model.pt", map_location="cpu")
model.load_state_dict(ckpt["state"])
model.to(device)
model.eval()
print(f"Loaded best model (epoch {ckpt['best_epoch']}, val macro-F1 {ckpt['best_val_f1']:.4f})")

# Inference on test
test_ds = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                        torch.tensor(y_test, dtype=torch.long))
test_loader = DataLoader(test_ds, batch_size=CFG["batch_size"], shuffle=False)
all_probs = []
with torch.no_grad():
    for emb, _ in test_loader:
        logits = model(emb.to(device))
        all_probs.append(torch.softmax(logits, dim=-1).cpu().numpy())
probs = np.concatenate(all_probs)
y_pred = probs.argmax(axis=1)

# Metrics
acc = accuracy_score(y_test, y_pred)
macro_p = precision_score(y_test, y_pred, average="macro", zero_division=0)
macro_r = recall_score(y_test, y_pred, average="macro", zero_division=0)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
print(f"Accuracy: {acc:.4f} | MacroP: {macro_p:.4f} | MacroR: {macro_r:.4f} | MacroF1: {macro_f1:.4f}")

# Classification report
report = classification_report(y_test, y_pred, labels=[0, 1, 2],
                               target_names=CFG["class_names"], digits=4, zero_division=0)
print("\nClassification report:\n", report)

# Confusion matrices
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=CFG["class_names"], yticklabels=CFG["class_names"])
axes[0].set_title("Confusion Matrix (counts)")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", ax=axes[1],
            xticklabels=CFG["class_names"], yticklabels=CFG["class_names"])
axes[1].set_title("Confusion Matrix (normalized)")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Actual")
plt.tight_layout()
plt.savefig("results/E0/confusion_matrix.png", dpi=150)
plt.show()
print("Saved confusion matrix -> results/E0/confusion_matrix.png")

## 8. Error Analysis and Visualization

Identify misclassified examples (with removed emoji info), plot training curves, and save the error table.

In [ ]:
# Cell: Error analysis + training curves
# Attach predictions to test dataframe
test_out = test_df.copy()
test_out["predicted_label"] = y_pred
test_out["pred_conf_bearish"] = probs[:, 0]
test_out["pred_conf_neutral"] = probs[:, 1]
test_out["pred_conf_bullish"] = probs[:, 2]
test_out["correct"] = (y_test == y_pred)
test_out.to_csv("results/E0/predictions.csv", index=False)

# Error analysis: misclassified examples
errors = test_out[~test_out["correct"]].copy()
print(f"Misclassified: {len(errors)} / {len(test_out)}")
errors.to_csv("results/E0/error_analysis.csv", index=False)
print("Saved error analysis -> results/E0/error_analysis.csv")

# Show a few error examples with emojis that were removed
print("\nSample error cases (with removed emoji info):")
for _, row in errors.sort_values("pred_conf_bullish", ascending=False).head(10).iterrows():
    print(f"  [{row['label_name']}->{CFG['class_names'][row['predicted_label']]}] {row['text_without_emoji'][:50]!r} | emojis={row['emoji_list']}")

# Training curves
epochs_arr = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(epochs_arr, history["train_loss"], marker="o", label="Train Loss")
axes[0].plot(epochs_arr, history["val_loss"], marker="o", label="Val Loss")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(epochs_arr, history["val_acc"], marker="o", color="green")
axes[1].set_title("Validation Accuracy"); axes[1].set_xlabel("Epoch")
axes[2].plot(epochs_arr, history["val_f1"], marker="o", color="purple")
axes[2].set_title("Validation Macro-F1"); axes[2].set_xlabel("Epoch")
plt.tight_layout()
plt.savefig("results/E0/training_history.png", dpi=150)
plt.show()
print("Saved training curves -> results/E0/training_history.png")

# Confidence distribution
plt.figure(figsize=(8, 5))
sns.histplot(probs.max(axis=1), bins=50)
plt.title("Prediction Confidence Distribution")
plt.xlabel("Max softmax probability")
plt.savefig("results/E0/confidence_distribution.png", dpi=150)
plt.show()

## 8b. Segmented Performance (short/medium/long + emoji count)

Report E0 test performance separately by text length and by original emoji count (post-hoc, using stored emoji metadata).

In [ ]:
# Cell: Segmented performance analysis
segmentation = {}

# 1. By text length (token_count)
bins = [-1, 5, 15, 10**9]
labels_bucket = ["short", "medium", "long"]
test_out["bucket"] = pd.cut(test_out["token_count"], bins=bins, labels=labels_bucket)

print("=== Performance by text length ===")
for b in labels_bucket:
    sub = test_out[test_out["bucket"] == b]
    if len(sub):
        s_acc = accuracy_score(sub["label"], sub["predicted_label"])
        s_f1 = f1_score(sub["label"], sub["predicted_label"], average="macro", zero_division=0)
        segmentation[f"bucket_{b}"] = {"n": int(len(sub)), "acc": float(s_acc), "macro_f1": float(s_f1)}
        print(f"  {b:7s}: n={len(sub):5d} acc={s_acc:.4f} macroF1={s_f1:.4f}")

# 2. By original emoji count
print("\n=== Performance by original emoji count ===")
test_out["emoji_bucket"] = test_out["num_emojis"].apply(lambda x: "3+" if x >= 3 else str(x))
for ec in ["1", "2", "3+"]:
    sub = test_out[test_out["emoji_bucket"] == ec]
    if len(sub):
        s_acc = accuracy_score(sub["label"], sub["predicted_label"])
        s_f1 = f1_score(sub["label"], sub["predicted_label"], average="macro", zero_division=0)
        segmentation[f"emojis_{ec}"] = {"n": int(len(sub)), "acc": float(s_acc), "macro_f1": float(s_f1)}
        print(f"  {ec} emoji(s): n={len(sub):5d} acc={s_acc:.4f} macroF1={s_f1:.4f}")

# Save segmentation
with open("results/E0/segmentation.json", "w", encoding="utf-8") as f:
    json.dump(segmentation, f, indent=2)
print("\nSaved segmentation -> results/E0/segmentation.json")

## 9. Results Summary and Comparison

Compile all metrics to `metrics.json`, generate `E0_report.md`, and save a config snapshot for reproducibility.

In [ ]:
# Cell: Save results, metrics, config, and E0 report
# Dataset hash (reproducibility)
h = hashlib.sha1()
for text in train_df["text_without_emoji"].tolist():
    h.update(text.encode("utf-8"))
dataset_hash = h.hexdigest()

# Compile metrics
per_class = {}
for i, name in enumerate(CFG["class_names"]):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    per_class[name] = {"precision": float(p), "recall": float(r), "f1": float(f), "support": int(cm[i, :].sum())}

metrics = {
    "experiment": "E0",
    "input": "text_without_emoji",
    "accuracy": float(acc),
    "macro_precision": float(macro_p),
    "macro_recall": float(macro_r),
    "macro_f1": float(macro_f1),
    "per_class": per_class,
    "confusion_matrix": cm.tolist(),
    "confusion_matrix_normalized": cm_norm.tolist(),
    "best_epoch": int(best_epoch),
    "best_val_macro_f1": float(best_val_f1),
    "training_time_sec": float(training_time),
    "dataset_sizes": {"train": int(len(train_df)), "validation": int(len(val_df)), "test": int(len(test_df))},
    "class_weights": {name: float(class_weights[i]) for i, name in enumerate(CFG["class_names"])},
    "seed": CFG["seed"],
    "dataset_hash": dataset_hash,
    "limitations": [
        "Validation/test have ~7.2% exact-text wording overlap (865 texts); reported as a dataset limitation.",
        "Class imbalance toward Bullish; class-weighted loss used (train-only).",
        "Frozen BERT may limit capacity."
    ],
}
with open("results/E0/metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
print("Saved metrics.json")

# Config snapshot
with open("results/E0/config_snapshot.yaml", "w", encoding="utf-8") as f:
    import yaml
    yaml.safe_dump({"E0": CFG, "dataset_hash": dataset_hash}, f)
print("Saved config_snapshot.yaml")

# Generate E0 markdown report
per_class_rows = "".join(
    f"| {name} | {per_class[name]['precision']:.3f} | {per_class[name]['recall']:.3f} | {per_class[name]['f1']:.3f} |\n"
    for name in CFG["class_names"]
)
cm_rows = "".join(
    f"| {CFG['class_names'][i]} | " + " | ".join(str(int(cm[i][j])) for j in range(3)) + " |\n"
    for i in range(3)
)
seg_rows = "".join(
    f"| {k} | {v['n']} | {v['acc']:.4f} | {v['macro_f1']:.4f} |\n"
    for k, v in segmentation.items()
)

markdown = f"""# E0 — Text-Only Sentiment Baseline Report

## 1. Objective
Estimate how well sentiment is predicted from text **with emojis removed** (control condition for the emoji research question).

## 2. Dataset
Final approved splits:
- Train: {len(train_df)} (deduplicated) · Validation: {len(val_df)} · Test: {len(test_df)}
- Classes: 0=Bearish, 1=Neutral, 2=Bullish. E0 input is `text_without_emoji`.

## 3. Preprocessing
Emojis removed with Unicode-aware `emoji` library; `text_without_emoji` used as input. BERT tokenizer (max_length 128).

## 4. Model Architecture
Frozen `bert-base-uncased` → mean pooling → MLP head ({CFG['hidden_size']}→{CFG['classifier_hidden']}→3) with dropout {CFG['dropout']}.

## 5. Training Setup
Seed {CFG['seed']} · Class-weighted CE (weights: { {CFG['class_names'][i]: round(class_weights[i],3) for i in range(3)} }) · AdamW (lr={CFG['learning_rate']}, wd={CFG['weight_decay']}) · Batch {CFG['batch_size']} · Epochs {CFG['epochs']} · Early stop patience 5 · Best by val macro-F1 (epoch {best_epoch}). Time: {training_time:.1f}s.

## 6. Results (Test)
| Metric | Value |
|---|---|
| Accuracy | {acc:.4f} |
| Macro Precision | {macro_p:.4f} |
| Macro Recall | {macro_r:.4f} |
| **Macro F1** | {macro_f1:.4f} |

Per-class:
| Class | Precision | Recall | F1 |
|---|---|---|---|
{per_class_rows}

## 7. Confusion Matrix (counts)
| Actual \\ Predicted | Bearish | Neutral | Bullish |
|---|---|---|---|
{cm_rows}

## 8. Segmented Performance
| Segment | n | Acc | Macro-F1 |
|---|---|---|---|
{seg_rows}

## 9. Error Analysis
See `results/E0/error_analysis.csv` (misclassified examples with removed emoji info).

## 10. Observations & Limitations
- Heavy Bullish imbalance; class weights from train only.
- Validation/test ~7.2% wording overlap documented as a limitation.
- Frozen BERT limits capacity.
- No conclusion about emoji value yet — requires E1-E5.
"""
with open("results/E0/E0_report.md", "w", encoding="utf-8") as f:
    f.write(markdown)
print("Saved E0_report.md")

print("\n=== E0 COMPLETE ===")
print(f"Test Accuracy: {acc:.4f} | Test Macro-F1: {macro_f1:.4f}")
print("All artifacts in results/E0/")